In [ ]:
import os
import glob
import zipfile
import re

# re = string and text manipulation

# glob = pattern matching tool, using like *txt

# Structure: UT/raw (for zips) and UT/ifgramStack (for extracted matrices)
PROJECT_NAME = "UT"

workspace_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
project_dir = os.path.join(workspace_dir, "data", PROJECT_NAME)

zip_dir = os.path.join(project_dir, "raw")
out_dir = os.path.join(project_dir, "inputs", "ifgramStack")

print(f"[*] Scanning {zip_dir} for HyP3 zip files...")
os.makedirs(out_dir, exist_ok=True)

zip_files = glob.glob(os.path.join(zip_dir, "*.zip"))
total_zips = len(zip_files)

if total_zips == 0:
    print(f"[!] FATAL: No .zip files found in {zip_dir}.")
    print(f"    Make sure your downloaded files are mapped to data/{PROJECT_NAME}/raw/")
else:
    print(f"[*] FOUND {total_zips} ZIP FILES.")
    print(f"[*] Extracting structurally to: {out_dir} \n")

    # Iterate through each zip
    for i, zf in enumerate(zip_files, 1):
        basename = os.path.basename(zf)

        # Parse Dates from Filename (HyP3 format: S1AA_20250805T123456_20250910T123456...)
        matches = re.findall(r'(\d{8})T\d{6}', basename)
        if len(matches) < 2:
            print(f"[{i}/{total_zips}] [!] Missing valid dates in filename: {basename}. Skipping.")
            continue

        ref_date, sec_date = matches[0], matches[1]
        pair_folder_name = f"{ref_date}_{sec_date}"

        # Create the specific pair directory MintPy needs
        pair_dir = os.path.join(out_dir, pair_folder_name)
        os.makedirs(pair_dir, exist_ok=True)

        print(f"[{i}/{total_zips}] Processing Pair: {pair_folder_name}")

        try:
            with zipfile.ZipFile(zf, 'r') as z:
                namelist = z.namelist()

                # Define exactly what 4 geometric/physics files we need + the config text file!
                required_targets = {
                    'Phase': ['unw_phase.tif'],
                    'Coherence': ['corr.tif', 'coh.tif'],
                    'DEM': ['dem.tif'],
                    'Incidence angle': ['lv_theta.tif', 'inc_map.tif'],
                    'Metadata': ['.txt']
                }

                # Extract only the targeted files into the pair directory
                for tag, suffixes in required_targets.items():
                    # For metadata, ensure we don't accidentally match another txt file by making sure it's the main file
                    if tag == 'Metadata':
                        matched_files = [f for f in namelist if f.endswith('.txt') and 'README' not in f and 'parameters' not in f]
                    else:
                        matched_files = [f for f in namelist if any(f.endswith(s) for s in suffixes)]

                    if matched_files:
                        target_file_in_zip = matched_files[0]
                        out_file_name = os.path.basename(target_file_in_zip)
                        out_file_path = os.path.join(pair_dir, out_file_name)

                        if not os.path.exists(out_file_path):
                            print(f"    -> Extracting {tag:15} | {out_file_name}")
                            with z.open(target_file_in_zip) as source, open(out_file_path, "wb") as target:
                                target.write(source.read())
                        else:
                            pass
                    else:
                        print(f"    [!] Missing {tag} file in {basename}")

        except zipfile.BadZipFile:
            print(f"    [!] FATAL: {basename} is corrupted. Please re-download.")

    print(f"\n[+] STACK ARCHITECTURE COMPLETE.")
    print(f"[*] Ready for MintPy config. View tree in: {out_dir}")